# Round42 Full Workflow Notebook: `v42_5_ta60_ec60_challenger_push.csv`

这个 notebook 展示平台期最激进但仍可解释的一类 challenger：把 TA/EC 同时推进到 `0.60 / 0.60`，同时在湿月样本上把 EC 放松回 `0.55`。

**为什么这本 notebook 重要**
- 它代表项目在 plateau 上继续探索外边界，而不是只重复 control。
- 它延续了 round37 的 corridor 思路，但把系数推得更远。
- 这个阶段也清楚说明了：后期很多 tied-best 文件其实是在探索边界安全范围。


## Step 1. Experiment Objective

**这个步骤做什么**
说明 round42 challenger-push 的设计目标。

**为什么要做这个步骤**
平台期 notebook 的价值，在于把“为什么继续试”讲清楚。

**预期产出**
得到该分支在 plateau 上的角色说明。


**本分支摘要**
- Output file: `v42_5_ta60_ec60_challenger_push.csv`
- TA alpha: `0.60`
- EC alpha: `0.60`
- Wet-month relax EC alpha: `0.55`
- Base source: `round29 legacy control`
- Delta source: `round37 control replay`


## Step 2. Environment Setup

**这个步骤做什么**
导入依赖并定义官方数据和公开资产地址。

**为什么要做这个步骤**
确保 notebook 从第一步开始就完整自洽。

**预期产出**
得到统一环境和远程读取函数。


In [ ]:
# =========================
# Step: Environment Setup
# 作用：导入依赖，定义官方数据源与公开 repo 资产地址
# =========================

from pathlib import Path
from io import StringIO
import subprocess

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

pd.set_option('display.max_columns', 220)
pd.set_option('display.width', 220)

OFFICIAL_RAW = 'https://raw.githubusercontent.com/Snowflake-Labs/EY-AI-and-Data-Challenge/refs/heads/main'
SHOWCASE_RAW = 'https://raw.githubusercontent.com/FlalaGoGoGo/ey-water-2026-showcase/main'
OUTPUT_DIR = Path('generated_outputs')
OUTPUT_DIR.mkdir(exist_ok=True)


def read_csv_remote(url: str) -> pd.DataFrame:
    """优先 pandas 直读，失败时回退到 curl，保证 notebook 更稳。"""
    try:
        return pd.read_csv(url)
    except Exception as exc:
        print(f'[warn] pandas direct read failed: {url}')
        print(f'       reason: {exc}')
        res = subprocess.run(['curl', '-L', url], check=True, capture_output=True, text=True)
        return pd.read_csv(StringIO(res.stdout))


## Step 3. Load Raw Data, CHIRPS Lookup, And Anchors

**这个步骤做什么**
读取官方原始数据、CHIRPS lookup，以及构造 round42 所需的两个公开锚点。

**为什么要做这个步骤**
这一步决定了平台期 challenger 是建立在什么基准之上的。

**预期产出**
得到 round42 重建所需的全部输入。


In [ ]:
train_wq = read_csv_remote(f'{OFFICIAL_RAW}/water_quality_training_dataset.csv')
val_tpl = read_csv_remote(f'{OFFICIAL_RAW}/submission_template.csv')
chirps = read_csv_remote(f'{SHOWCASE_RAW}/notebooks/assets/cache/chirps_monthly_lookup.csv')
legacy_control = read_csv_remote(f'{SHOWCASE_RAW}/notebooks/assets/anchors/round29/v29_5_static_hydro_ood_guard.csv')
control_30 = read_csv_remote(f'{SHOWCASE_RAW}/notebooks/assets/anchors/round37/v37_1_control_v36_1.csv')
reference_target = read_csv_remote(f'{SHOWCASE_RAW}/notebooks/assets/targets/v42_5_ta60_ec60_challenger_push.csv')

print('train_wq:', train_wq.shape)
print('val_tpl:', val_tpl.shape)
print('chirps:', chirps.shape)
print('legacy_control:', legacy_control.shape)
print('control_30:', control_30.shape)


## Step 4. Build Consensus Deltas And Wet-Month Mask

**这个步骤做什么**
同时构造 TA/EC 正向共识增量和 CHIRPS 湿月门控。

**为什么要做这个步骤**
round42 的关键就是“更大外边界 + 轻门控回退”这两个部件一起工作。

**预期产出**
得到 challenger 渲染所需的 delta 和 wet mask。


In [ ]:
def add_year_month(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    out['Sample Date'] = pd.to_datetime(out['Sample Date'], dayfirst=True, errors='coerce')
    out['year'] = out['Sample Date'].dt.year.astype(int)
    out['month'] = out['Sample Date'].dt.month.astype(int)
    return out

base_ta = legacy_control['Total Alkalinity'].to_numpy(dtype=float)
base_ec = legacy_control['Electrical Conductance'].to_numpy(dtype=float)
base_drp = legacy_control['Dissolved Reactive Phosphorus'].to_numpy(dtype=float)
control_ta = control_30['Total Alkalinity'].to_numpy(dtype=float)
control_ec = control_30['Electrical Conductance'].to_numpy(dtype=float)
ta_consensus_delta = np.maximum(control_ta - base_ta, 0.0) / 0.30
ec_consensus_delta = np.maximum(control_ec - base_ec, 0.0) / 0.30

site_month_stats = (
    chirps.groupby(['Latitude', 'Longitude', 'month'])['chirps_monthly_precip']
    .agg(['mean', 'std'])
    .reset_index()
    .rename(columns={'mean': 'chirps_site_month_mean', 'std': 'chirps_site_month_std'})
)
train = add_year_month(train_wq).merge(chirps, on=['Latitude', 'Longitude', 'year', 'month'], how='left')
train = train.merge(site_month_stats, on=['Latitude', 'Longitude', 'month'], how='left')
train['chirps_site_month_std'] = train['chirps_site_month_std'].replace(0.0, np.nan)
train['chirps_anom'] = train['chirps_monthly_precip'] - train['chirps_site_month_mean']
train['chirps_z'] = train['chirps_anom'] / train['chirps_site_month_std']
wet_threshold = float(train['chirps_z'].dropna().quantile(0.80))

sub = add_year_month(val_tpl).merge(chirps, on=['Latitude', 'Longitude', 'year', 'month'], how='left')
sub = sub.merge(site_month_stats, on=['Latitude', 'Longitude', 'month'], how='left')
sub['chirps_site_month_std'] = sub['chirps_site_month_std'].replace(0.0, np.nan)
sub['chirps_anom'] = sub['chirps_monthly_precip'] - sub['chirps_site_month_mean']
sub['chirps_z'] = sub['chirps_anom'] / sub['chirps_site_month_std']
wet_mask = (sub['chirps_z'] >= wet_threshold).fillna(False).to_numpy()

print('wet_threshold_train_q80:', round(wet_threshold, 4))
print('wet_row_count:', int(wet_mask.sum()))


## Step 5. Render The 60/60 Challenger Push

**这个步骤做什么**
把 TA 和 EC 推到 0.60 / 0.60，同时在湿月把 EC 放松回 0.55。

**为什么要做这个步骤**
这是平台期最典型的“push with guard”思路。

**预期产出**
得到 round42 challenger-push 的最终预测。


In [ ]:
ta = np.clip(base_ta + 0.60 * ta_consensus_delta, 0, None)
ec60 = np.clip(base_ec + 0.60 * ec_consensus_delta, 0, None)
ec55 = np.clip(base_ec + 0.55 * ec_consensus_delta, 0, None)
ec = ec60.copy()
ec[wet_mask] = ec55[wet_mask]
drp = np.clip(base_drp, 0, None)

submission = val_tpl[['Latitude', 'Longitude', 'Sample Date']].copy()
submission['Sample Date'] = pd.to_datetime(submission['Sample Date'], dayfirst=True, errors='coerce').dt.strftime('%d/%m/%Y')
submission['Total Alkalinity'] = ta
submission['Electrical Conductance'] = ec
submission['Dissolved Reactive Phosphorus'] = drp
submission = submission[['Latitude', 'Longitude', 'Sample Date', 'Total Alkalinity', 'Electrical Conductance', 'Dissolved Reactive Phosphorus']]
display(submission.head(3))


## Step 6. Export And Verify

**这个步骤做什么**
导出最终 csv，并与公开参考文件逐列比较。

**为什么要做这个步骤**
公开 notebook 需要证明自己生成的是正确目标文件。

**预期产出**
得到输出文件和 exact-match 复现结果。


In [ ]:
out_path = OUTPUT_DIR / 'v42_5_ta60_ec60_challenger_push.csv'
submission.to_csv(out_path, index=False)
print('saved to:', out_path.resolve())

metric_cols = ['Total Alkalinity', 'Electrical Conductance', 'Dissolved Reactive Phosphorus']
diff = (submission[metric_cols] - reference_target[metric_cols]).abs()
print('max abs diff by target:')
print(diff.max())
print('exact match:', bool((diff.max() < 1e-12).all()))


## Step 7. Diagnostics And Interpretation

**这个步骤做什么**
量化 challenger 相对 legacy control 的位移，以及湿月回退影响的行数。

**为什么要做这个步骤**
这样外部读者就能区分“push 的主体”与“guard 的范围”。

**预期产出**
得到 round42 challenger 的风险与位移摘要。


In [ ]:
changed_ec_mask = np.abs(ec60 - ec) > 1e-12
summary = pd.DataFrame({
    'metric': ['TA vs base', 'EC vs base', 'DRP vs base', 'Wet relaxed EC rows'],
    'value': [
        float(np.mean(np.abs(ta - base_ta))),
        float(np.mean(np.abs(ec - base_ec))),
        float(np.mean(np.abs(drp - base_drp))),
        int(changed_ec_mask.sum()),
    ]
})
display(summary)

plt.figure(figsize=(7, 4))
plt.hist(sub['chirps_z'].dropna(), bins=28, color='#ffd166', edgecolor='black', alpha=0.85)
plt.axvline(wet_threshold, color='#ffe600', linestyle='--', label='wet threshold')
plt.title('Round42 validation CHIRPS z-score distribution')
plt.legend()
plt.show()


## Step 8. Interpretation

**这个步骤做什么**
把 round42 challenger-push 放回平台期策略中解释。

**为什么要做这个步骤**
平台期最容易被误解成“重复提交”，但其实很多 tied-best 版本是在确认边界安全范围。

**本分支结论**
- `0.60 / 0.60` challenger 把 round37 的 corridor 推到了更远的外边界。
- CHIRPS 门控只在湿月对 EC 做回退，因此风险控制仍然清晰。
- 这本 notebook 体现了后期 plateau 阶段的典型思路：先 push，再用小门控把最危险的点收回来。
